# 1D Cross-Section Levee Authoring

Read, modify, insert, and remove cross-section levee station-elevation data
in a real HEC-RAS geometry file, then run the unsteady simulation on the
modified geometry and verify results extraction from the HDF output.

**Example Project:** Bald Eagle Creek (1D Unsteady Flow Hydraulics)

## Overview

HEC-RAS 1D cross sections can have left and/or right levee points defined as station-elevation pairs. These control when water overtops the levee and enters the floodplain.

**Storage format:** Each cross section's levee data is stored on a single `Levee=` line in the `.g##` text file:
```
Levee=left_flag,left_station,left_elevation,right_flag,right_station,right_elevation,0,0
```
- Active sides use flag `-1` with station and elevation values
- Inactive sides use flag `0` with blank station/elevation fields

**API methods:**
- `get_levees()` — Read all levee data as a DataFrame
- `set_levees()` — Write levee data for a single cross section (update or insert)

## 1. Setup and Imports

In [ ]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
USE_LOCAL_SOURCE = False

if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path

    cwd = Path.cwd()
    local_path = cwd if (cwd / "ras_commander").exists() else cwd.parent
    if str(local_path) not in sys.path:
        sys.path.insert(0, str(local_path))
    print(f"LOCAL SOURCE MODE: loading from {local_path / 'ras_commander'}")
else:
    print("PIP PACKAGE MODE: loading installed ras-commander")

from pathlib import Path
import logging
import os
import re
import warnings

import pandas as pd
import numpy as np
import shutil

from IPython.display import display

from ras_commander import RasExamples
from ras_commander.geom.GeomCrossSection import GeomCrossSection

warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("ras_commander").setLevel(logging.CRITICAL)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

import ras_commander

print(f"Loaded: {ras_commander.__file__}")

## 2. Extract Example Project

In [2]:
project_path = RasExamples.extract_project("Balde Eagle Creek")
geom_file = project_path / "BaldEagle.g01"

print(f"Project: {project_path}")
print(f"Geometry file exists: {geom_file.exists()}")

2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - Found zip file: C:\Users\bill\AppData\Local\ras-commander\examples\Example_Projects_7_0.zip


2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - Loading project data from CSV...


2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - Loaded 67 projects from CSV.


2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - ----- RasExamples Extracting Project -----


2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - Extracting project 'Balde Eagle Creek'


2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - Folder 'Balde Eagle Creek' already exists. Deleting existing folder...


2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - Existing folder 'Balde Eagle Creek' has been deleted.


2026-05-08 13:00:07 - ras_commander.RasExamples - INFO - Successfully extracted project 'Balde Eagle Creek' to G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek


Project: G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek
Geometry file exists: True


## 3. Read All Levees

`get_levees()` returns a DataFrame with columns: `xs_id`, `River`, `Reach`, `RS`, `left_station`, `left_elevation`, `right_station`, `right_elevation`. Cross sections without levees have NaN values.

In [3]:
all_levees = GeomCrossSection.get_levees(geom_file)

total_xs = len(all_levees)
has_left = all_levees["left_station"].notna().sum()
has_right = all_levees["right_station"].notna().sum()
has_both = (all_levees["left_station"].notna() & all_levees["right_station"].notna()).sum()
has_none = (all_levees["left_station"].isna() & all_levees["right_station"].isna()).sum()

print(f"Total cross sections: {total_xs}")
print(f"  With left levee:  {has_left}")
print(f"  With right levee: {has_right}")
print(f"  With both sides:  {has_both}")
print(f"  With no levees:   {has_none}")
print(f"\nColumns: {list(all_levees.columns)}")
print(f"\nFirst 15 rows:")
all_levees.head(15)

2026-05-08 13:00:07 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 189 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle.g01


Total cross sections: 189
  With left levee:  28
  With right levee: 18
  With both sides:  5
  With no levees:   148

Columns: ['xs_id', 'River', 'Reach', 'RS', 'left_station', 'left_elevation', 'right_station', 'right_elevation']

First 15 rows:


,xs_id,River,Reach,RS,left_station,left_elevation,right_station,right_elevation
0,Bald Eagle|Loc Hav|138154.4,Bald Eagle,Loc Hav,138154.4,NaN,NaN,NaN,NaN
1,Bald Eagle|Loc Hav|137690.8,Bald Eagle,Loc Hav,137690.8,NaN,NaN,NaN,NaN
2,Bald Eagle|Loc Hav|137327.0,Bald Eagle,Loc Hav,137327.0,NaN,NaN,NaN,NaN
3,Bald Eagle|Loc Hav|136564.9,Bald Eagle,Loc Hav,136564.9,NaN,NaN,NaN,NaN
4,Bald Eagle|Loc Hav|136202.3,Bald Eagle,Loc Hav,136202.3,NaN,NaN,NaN,NaN
5,Bald Eagle|Loc Hav|135591.4,Bald Eagle,Loc Hav,135591.4,NaN,NaN,574.99,NaN
6,Bald Eagle|Loc Hav|135068.7,Bald Eagle,Loc Hav,135068.7,NaN,NaN,NaN,NaN
7,Bald Eagle|Loc Hav|134487.2,Bald Eagle,Loc Hav,134487.2,NaN,NaN,NaN,NaN
8,Bald Eagle|Loc Hav|133881.0,Bald Eagle,Loc Hav,133881.0,NaN,NaN,NaN,NaN
9,Bald Eagle|Loc Hav|133446.1,Bald Eagle,Loc Hav,133446.1,NaN,NaN,NaN,NaN


## 4. Filter by River/Reach/RS

Use optional `river`, `reach`, and `rs` parameters to narrow the query.

In [4]:
river_name = all_levees["River"].iloc[0]
reach_name = all_levees["Reach"].iloc[0]

print(f"River: {river_name}, Reach: {reach_name}")

# Filter to just cross sections with active levees
active_levees = all_levees[
    all_levees["left_station"].notna() | all_levees["right_station"].notna()
].copy()

print(f"\nActive levee cross sections ({len(active_levees)} of {total_xs}):")
active_levees[["xs_id", "left_station", "left_elevation", "right_station", "right_elevation"]]

River: Bald Eagle, Reach: Loc Hav

Active levee cross sections (41 of 189):


,xs_id,left_station,left_elevation,right_station,right_elevation
5,Bald Eagle|Loc Hav|135591.4,NaN,NaN,574.99,NaN
10,Bald Eagle|Loc Hav|132973.6,80.0,NaN,NaN,NaN
14,Bald Eagle|Loc Hav|130339.2,100.0,NaN,NaN,NaN
18,Bald Eagle|Loc Hav|127410.9,NaN,NaN,2349.99,NaN
19,Bald Eagle|Loc Hav|126741.0,NaN,NaN,1935.00,NaN
52,Bald Eagle|Loc Hav|104647.2,NaN,NaN,3488.00,NaN
53,Bald Eagle|Loc Hav|104195.0,NaN,NaN,3068.00,NaN
54,Bald Eagle|Loc Hav|103854.0,NaN,NaN,3285.00,NaN
55,Bald Eagle|Loc Hav|103369.7,205.0,NaN,3615.00,NaN
57,Bald Eagle|Loc Hav|103122.3,135.0,NaN,3630.00,NaN


In [5]:
# Read a single cross section by xs_id
target_xs = active_levees["xs_id"].iloc[0]
single = GeomCrossSection.get_levees(geom_file, xs_id=target_xs)

print(f"Single cross section: {target_xs}")
single

2026-05-08 13:00:07 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 1 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle.g01


Single cross section: Bald Eagle|Loc Hav|135591.4


,xs_id,River,Reach,RS,left_station,left_elevation,right_station,right_elevation
0,Bald Eagle|Loc Hav|135591.4,Bald Eagle,Loc Hav,135591.4,NaN,NaN,574.99,NaN


## 5. Update Existing Levees (Round-Trip)

`set_levees()` updates the `Levee=` line for a specific cross section. We demonstrate a read-modify-write round trip: raise the left levee elevation by 2 feet, write it back, and verify.

In [6]:
# Create a working copy
work_copy = project_path / "BaldEagle_levee_test.g01"
shutil.copy(geom_file, work_copy)
print(f"Working copy: {work_copy.name}")

# Pick a cross section with both-side levees
both_sides = active_levees[
    active_levees["left_station"].notna() & active_levees["right_station"].notna()
]
target_xs = both_sides["xs_id"].iloc[0]
original = GeomCrossSection.get_levees(work_copy, xs_id=target_xs)

print(f"\nOriginal levees for {target_xs}:")
print(f"  Left:  station={original['left_station'].iloc[0]}")
print(f"  Right: station={original['right_station'].iloc[0]}")

# Move the left levee station outward by 10 units
new_left_sta = original["left_station"].iloc[0] + 10.0

backup = GeomCrossSection.set_levees(
    work_copy,
    xs_id=target_xs,
    left_station=new_left_sta,
    right_station=original["right_station"].iloc[0],
)
print(f"\nBackup: {backup.name}")

# Verify round trip
updated = GeomCrossSection.get_levees(work_copy, xs_id=target_xs)
print(f"\nUpdated levees:")
print(f"  Left:  station={updated['left_station'].iloc[0]}")
print(f"  Right: station={updated['right_station'].iloc[0]}")

assert np.isclose(updated["left_station"].iloc[0], new_left_sta), "Left station mismatch!"
assert np.isclose(updated["right_station"].iloc[0], original["right_station"].iloc[0]), "Right station changed!"
print(f"\n✓ Round trip verified")

Working copy: BaldEagle_levee_test.g01


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 1 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01



Original levees for Bald Eagle|Loc Hav|103369.7:
  Left:  station=205.0
  Right: station=3615.0


2026-05-08 13:00:08 - ras_commander.geom.GeomParser - INFO - Created backup: G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01.bak


2026-05-08 13:00:08 - ras_commander.geom.GeomParser - INFO - Successfully wrote geometry file: G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Updated levees for Bald Eagle/Loc Hav/RS 103369.7


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 1 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01



Backup: BaldEagle_levee_test.g01.bak

Updated levees:
  Left:  station=215.0
  Right: station=3615.0

✓ Round trip verified


## 6. Insert Levees into a Cross Section Without Them

When a cross section has no `Levee=` line, `set_levees()` inserts one after the Manning's n data block.

In [7]:
# Find a cross section with no levees
all_work = GeomCrossSection.get_levees(work_copy)
no_levees = all_work[
    all_work["left_station"].isna() & all_work["right_station"].isna()
]

insert_target = no_levees["xs_id"].iloc[0]
print(f"Cross section with no levees: {insert_target}")

# Insert a right-side-only levee (station + elevation)
GeomCrossSection.set_levees(
    work_copy,
    xs_id=insert_target,
    right_station=500.0,
    right_elevation=650.0,
    create_backup=False,
)

# Verify insertion
inserted = GeomCrossSection.get_levees(work_copy, xs_id=insert_target)
print(f"\nAfter insertion:")
print(f"  Left:  station={inserted['left_station'].iloc[0]} (inactive)")
print(f"  Right: station={inserted['right_station'].iloc[0]}, elevation={inserted['right_elevation'].iloc[0]}")

assert np.isnan(inserted["left_station"].iloc[0]), "Left should be inactive"
assert np.isclose(inserted["right_station"].iloc[0], 500.0), "Right station mismatch"
assert np.isclose(inserted["right_elevation"].iloc[0], 650.0), "Right elevation mismatch"
print(f"✓ Right-only levee inserted successfully")

2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 189 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01


2026-05-08 13:00:08 - ras_commander.geom.GeomParser - INFO - Successfully wrote geometry file: G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Updated levees for Bald Eagle/Loc Hav/RS 138154.4


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 1 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01


Cross section with no levees: Bald Eagle|Loc Hav|138154.4

After insertion:
  Left:  station=nan (inactive)
  Right: station=500.0, elevation=650.0
✓ Right-only levee inserted successfully


## 7. Remove a Levee Side

Omit station/elevation for a side to deactivate it. Here we take a cross section with both sides and deactivate the left.

In [8]:
# Pick a cross section that currently has both sides
current = GeomCrossSection.get_levees(work_copy)
both_now = current[
    current["left_station"].notna() & current["right_station"].notna()
]
deactivate_target = both_now["xs_id"].iloc[0]

before = GeomCrossSection.get_levees(work_copy, xs_id=deactivate_target)
print(f"Before (both sides active):")
print(f"  Left:  station={before['left_station'].iloc[0]}")
print(f"  Right: station={before['right_station'].iloc[0]}")

# Keep only the right side
GeomCrossSection.set_levees(
    work_copy,
    xs_id=deactivate_target,
    right_station=before["right_station"].iloc[0],
    create_backup=False,
)

after = GeomCrossSection.get_levees(work_copy, xs_id=deactivate_target)
print(f"\nAfter (left deactivated):")
print(f"  Left:  station={after['left_station'].iloc[0]}")
print(f"  Right: station={after['right_station'].iloc[0]}")

assert np.isnan(after["left_station"].iloc[0]), "Left should now be inactive"
assert np.isclose(after["right_station"].iloc[0], before["right_station"].iloc[0]), "Right station changed"
print(f"✓ Left levee deactivated, right preserved")

2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 189 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 1 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01


Before (both sides active):
  Left:  station=215.0
  Right: station=3615.0


2026-05-08 13:00:08 - ras_commander.geom.GeomParser - INFO - Successfully wrote geometry file: G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Updated levees for Bald Eagle/Loc Hav/RS 103369.7


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 1 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle_levee_test.g01



After (left deactivated):
  Left:  station=nan
  Right: station=3615.0
✓ Left levee deactivated, right preserved


## 8. Identify by River/Reach/RS Instead of xs_id

Both `get_levees()` and `set_levees()` accept `river`, `reach`, and `rs` parameters as an alternative to `xs_id`.

In [9]:
# Pick a cross section and use river/reach/rs addressing
sample = active_levees.iloc[1]
river_val = sample["River"]
reach_val = sample["Reach"]
rs_val = sample["RS"]

print(f"Querying by river={river_val}, reach={reach_val}, rs={rs_val}")

by_rrs = GeomCrossSection.get_levees(
    geom_file,
    river=river_val,
    reach=reach_val,
    rs=rs_val,
)
print(f"\nResult:")
by_rrs

Querying by river=Bald Eagle, reach=Loc Hav, rs=132973.6


2026-05-08 13:00:08 - ras_commander.geom.GeomCrossSection - INFO - Read levee data for 1 cross sections from G:\GH\ras-commander\examples\example_projects\Balde Eagle Creek\BaldEagle.g01



Result:


,xs_id,River,Reach,RS,left_station,left_elevation,right_station,right_elevation
0,Bald Eagle|Loc Hav|132973.6,Bald Eagle,Loc Hav,132973.6,80.0,NaN,NaN,NaN


## 9. Batch Analysis: Levee Elevation Summary

Use the DataFrame output for bulk analysis across the entire geometry.

In [10]:
# Compute summary statistics for active levees
active = all_levees[
    all_levees["left_station"].notna() | all_levees["right_station"].notna()
].copy()

print(f"Levee elevation statistics (active cross sections only):")
print(f"\nLeft levees ({active['left_station'].notna().sum()} active):")
if active["left_elevation"].notna().any():
    print(f"  Min elevation: {active['left_elevation'].min():.2f}")
    print(f"  Max elevation: {active['left_elevation'].max():.2f}")
    print(f"  Mean elevation: {active['left_elevation'].mean():.2f}")
    print(f"  Station range: {active['left_station'].min():.2f} to {active['left_station'].max():.2f}")

print(f"\nRight levees ({active['right_station'].notna().sum()} active):")
if active["right_elevation"].notna().any():
    print(f"  Min elevation: {active['right_elevation'].min():.2f}")
    print(f"  Max elevation: {active['right_elevation'].max():.2f}")
    print(f"  Mean elevation: {active['right_elevation'].mean():.2f}")
    print(f"  Station range: {active['right_station'].min():.2f} to {active['right_station'].max():.2f}")

Levee elevation statistics (active cross sections only):

Left levees (28 active):

Right levees (18 active):
  Min elevation: 544.39
  Max elevation: 544.39
  Mean elevation: 544.39
  Station range: 574.99 to 3945.00


## 10. Cross-Section Profiles With Levee Positions

Visualize representative cross sections before and after the levee modification that will be validated by the HEC-RAS compute in the next section.

In [ ]:
import matplotlib.pyplot as plt

# ── Select representative cross sections ─────────────────────────────────────
xs_left_only  = active_levees[
    active_levees["left_station"].notna() & active_levees["right_station"].isna()
].iloc[0]
xs_right_only = active_levees[
    active_levees["right_station"].notna() & active_levees["left_station"].isna()
].iloc[0]
_both = active_levees[
    active_levees["left_station"].notna() & active_levees["right_station"].notna()
]
xs_both_1 = _both.iloc[0]
xs_both_2 = _both.iloc[1]

# Cross section that Section 11 will modify (+15 ft left station)
modify_target = active_levees[active_levees["left_station"].notna()].iloc[0]
orig_left_sta = modify_target["left_station"]
new_left_sta  = orig_left_sta + 15.0

# ── Figure 1: Four representative cross sections with levee positions ─────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(
    "Representative Cross Sections With Levee Positions — Bald Eagle Creek",
    fontsize=13, fontweight="bold",
)

configs = [
    (xs_left_only,  "Left Levee Only"),
    (xs_right_only, "Right Levee Only"),
    (xs_both_1,     "Both Levees"),
    (xs_both_2,     "Both Levees (second example)"),
]

for ax, (row, title) in zip(axes.flat, configs):
    river_v = row["River"]
    reach_v = row["Reach"]
    rs_v    = str(row["RS"])
    try:
        se_df  = GeomCrossSection.get_station_elevation(geom_file, river_v, reach_v, rs_v)
        banks  = GeomCrossSection.get_bank_stations(geom_file, river_v, reach_v, rs_v)
        sta    = se_df["Station"].values
        elev   = se_df["Elevation"].values

        ax.fill_between(sta, elev.min() - 1, elev, alpha=0.15, color="saddlebrown")
        ax.plot(sta, elev, "k-", linewidth=1.5, label="Ground")

        if banks:
            lb, rb = banks
            ax.axvline(lb, color="royalblue", linestyle="--", linewidth=1.2,
                       label=f"L bank ({lb:.0f} ft)")
            ax.axvline(rb, color="steelblue", linestyle="--", linewidth=1.2,
                       label=f"R bank ({rb:.0f} ft)")
        if pd.notna(row["left_station"]):
            ax.axvline(row["left_station"], color="crimson", linewidth=2.5,
                       label=f"L levee ({row['left_station']:.0f} ft)")
        if pd.notna(row["right_station"]):
            lbl = f"R levee ({row['right_station']:.0f} ft)"
            if pd.notna(row.get("right_elevation")):
                lbl += f" elev={row['right_elevation']:.1f}"
            ax.axvline(row["right_station"], color="darkorange", linewidth=2.5, label=lbl)

        ax.set_title(f"{title} — RS {rs_v}", fontsize=9)
        ax.set_xlabel("Station (ft)", fontsize=8)
        ax.set_ylabel("Elevation (ft)", fontsize=8)
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(True, alpha=0.3)
    except Exception as e:
        ax.text(0.5, 0.5, f"Error: {e}", transform=ax.transAxes, ha="center")

plt.tight_layout()
plt.show()

# ── Figure 2: Before/After for the cross section being modified ──────────────
river_m  = modify_target["River"]
reach_m  = modify_target["Reach"]
rs_m     = str(modify_target["RS"])
se_m     = GeomCrossSection.get_station_elevation(geom_file, river_m, reach_m, rs_m)
banks_m  = GeomCrossSection.get_bank_stations(geom_file, river_m, reach_m, rs_m)
sta_m    = se_m["Station"].values
elev_m   = se_m["Elevation"].values

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(sta_m, elev_m.min() - 1, elev_m, alpha=0.15, color="saddlebrown")
ax.plot(sta_m, elev_m, "k-", linewidth=1.5, label="Ground profile")

if banks_m:
    lb_m, rb_m = banks_m
    ax.axvline(lb_m, color="royalblue", linestyle="--", linewidth=1.2,
               label=f"Left bank ({lb_m:.0f} ft)")
    ax.axvline(rb_m, color="steelblue", linestyle="--", linewidth=1.2,
               label=f"Right bank ({rb_m:.0f} ft)")

ax.axvline(orig_left_sta, color="crimson", linewidth=2.5,
           label=f"Left levee BEFORE ({orig_left_sta:.0f} ft)")
ax.axvline(new_left_sta, color="limegreen", linewidth=2.5, linestyle="--",
           label=f"Left levee AFTER +15 ft → ({new_left_sta:.0f} ft)")

# Annotation arrow
y_arrow = elev_m.min() + (elev_m.max() - elev_m.min()) * 0.1
ax.annotate(
    "",
    xy=(new_left_sta, y_arrow),
    xytext=(orig_left_sta, y_arrow),
    arrowprops=dict(arrowstyle="->", color="purple", lw=2),
)
ax.text(
    (orig_left_sta + new_left_sta) / 2,
    y_arrow + (elev_m.max() - elev_m.min()) * 0.04,
    "+15 ft",
    ha="center", color="purple", fontsize=10, fontweight="bold",
)

ax.set_title(
    f"Levee Modification Preview — RS {rs_m}\n"
    "Left levee station shifted outward +15 ft (validated by HEC-RAS compute in Section 11)",
    fontsize=10,
)
ax.set_xlabel("Station (ft)")
ax.set_ylabel("Elevation (ft)")
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Modified XS: {modify_target['xs_id']}")
print(f"  Left levee station: {orig_left_sta:.1f} ft → {new_left_sta:.1f} ft (+15 ft)")


In [ ]:
if work_copy.exists():
    work_copy.unlink()
    print(f"Removed demo working copy: {work_copy.name}")
for bak in project_path.glob("BaldEagle_levee_test.g01.bak*"):
    bak.unlink()
    print(f"Removed backup: {bak.name}")
print("Demo working copy cleaned up before validation.")

## 11. Validate Write Format With HEC-RAS Compute

The round-trip assertions above confirm Python can read back what it wrote,
but the ultimate validation is running HEC-RAS on the **modified** geometry
and confirming the simulation completes without errors.

We extract a **fresh project copy**, apply levee modifications to the main
geometry file (not a renamed copy), then run the unsteady simulation.

In [ ]:
from ras_commander import init_ras_project, RasCmdr, HdfResultsPlan

logging.getLogger("ras_commander").setLevel(logging.INFO)

# Extract a FRESH project copy for compute validation
WORK_DIR = Path(os.environ.get(
    "RAS_COMMANDER_WORKDIR",
    project_path.parent / "levee_validation",
))
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

fresh_project = RasExamples.extract_project("Balde Eagle Creek", output_path=WORK_DIR, suffix="validate")
fresh_geom = fresh_project / "BaldEagle.g01"
assert fresh_geom.exists()

# --------------------------------------------------------------------------
# Apply levee modifications: shift left levee station outward by 15 ft.
# --------------------------------------------------------------------------

fresh_levees = GeomCrossSection.get_levees(fresh_geom)
active_fresh = fresh_levees[fresh_levees["left_station"].notna()].copy()

target_xs_id = active_fresh["xs_id"].iloc[0]
original_levee = GeomCrossSection.get_levees(fresh_geom, xs_id=target_xs_id)

orig_left_sta = original_levee["left_station"].iloc[0]
new_left_sta = orig_left_sta + 15.0

GeomCrossSection.set_levees(
    fresh_geom,
    xs_id=target_xs_id,
    left_station=new_left_sta,
    create_backup=False,
)

verify_levee = GeomCrossSection.get_levees(fresh_geom, xs_id=target_xs_id)
assert np.isclose(verify_levee["left_station"].iloc[0], new_left_sta), "Left station mismatch"
print(f"Modified levee for {target_xs_id}: left station {orig_left_sta} -> {new_left_sta}")

# --------------------------------------------------------------------------
# Run HEC-RAS on the MODIFIED geometry.
# --------------------------------------------------------------------------

ras_val = init_ras_project(fresh_project, "7.0")

plans_using_g01 = ras_val.plan_df[ras_val.plan_df["Geom File"] == "01"]
assert not plans_using_g01.empty, "No plan found using geometry file .g01"
plan_number = plans_using_g01.iloc[0]["plan_number"]

plan_file_path = Path(plans_using_g01.iloc[0]["full_path"])
plan_text = plan_file_path.read_text()
plan_text = re.sub(r"Program Version=.*", f"Program Version={ras_val.ras_version}", plan_text)
plan_file_path.write_text(plan_text)

geom_text = fresh_geom.read_text()
geom_text = re.sub(r"Program Version=.*", f"Program Version={ras_val.ras_version}", geom_text)
fresh_geom.write_text(geom_text)
print(f"Updated plan {plan_number} Program Version to {ras_val.ras_version}")

print(f"Running Plan {plan_number} on MODIFIED geometry (levee station shift +15 ft)...")
RasCmdr.compute_plan(plan_number, ras_object=ras_val, force_geompre=True, num_cores=4)

hdf_path = Path(ras_val.plan_df.loc[
    ras_val.plan_df["plan_number"] == plan_number, "HDF_Results_Path"
].iloc[0])
assert hdf_path.exists(), f"HDF file not created for plan {plan_number}"

messages = HdfResultsPlan.get_compute_messages(hdf_path)
lines = messages.splitlines() if messages else []

errors = [l for l in lines if "ERROR" in l.upper()
          and "VOLUME ACCOUNTING" not in l.upper()
          and "ITERATIONS" not in l.upper()
          and "WSEL ERROR" not in l.upper()]

if errors:
    print(f"ERRORS ({len(errors)}):")
    for e in errors[:10]:
        print(f"  {e}")
    raise RuntimeError(f"HEC-RAS reported {len(errors)} error(s) with modified geometry")

print(f"")
print(f"{'='*70}")
print(f"VALIDATION RESULT")
print(f"{'='*70}")
print(f"  Geometry file:    {fresh_geom.name} (MODIFIED)")
print(f"  Levee change:     {target_xs_id} left station +15 ft")
print(f"  HEC-RAS version:  {ras_val.ras_version}")
print(f"  Plan {plan_number}:          Preprocessing + simulation COMPLETED")
print(f"  Format errors:    {len(errors)}")
print(f"  HDF results:      {hdf_path.name}")
print(f"{'='*70}")
print(f"  set_levees write format: VALIDATED (HEC-RAS accepted)")
print(f"{'='*70}")

## 12. Verify Results Extraction From HDF

The compute validation above proves HEC-RAS accepted the modified levee
geometry. Extract cross-section water surface results from the HDF output
to confirm the simulation produced valid hydraulic data at cross sections
with levees.

In [ ]:
from ras_commander.hdf import HdfResultsXsec

xsec_ts = HdfResultsXsec.get_xsec_timeseries(hdf_path)
print(f"Cross-section time series dataset:")
print(f"  Variables: {list(xsec_ts.data_vars)}")
print(f"  Dimensions: {dict(xsec_ts.dims)}")
print(f"  Cross sections: {xsec_ts.dims.get('cross_section', 'N/A')}")
print(f"  Time steps: {xsec_ts.dims.get('time', 'N/A')}")

# Locate the modified cross section
parts = target_xs_id.split("|")
modified_rs = parts[2] if len(parts) == 3 else target_xs_id

stations = xsec_ts.coords["Station"].values
rs_float = float(modified_rs)
station_matches = [i for i, s in enumerate(stations) if abs(float(s) - rs_float) < 1.0]

if station_matches:
    xs_idx = station_matches[0]
    ws = xsec_ts["Water_Surface"][:, xs_idx].values
    max_ws = float(xsec_ts["Maximum_Water_Surface"].values[xs_idx])
    max_flow = float(xsec_ts["Maximum_Flow"].values[xs_idx])

    print(f"")
    print(f"Results at modified levee cross section (RS {modified_rs}):")
    print(f"  Max water surface:  {max_ws:.2f} ft")
    fmt = f"{max_flow:,.1f}"
    print(f"  Max flow:           {fmt} cfs")
    print(f"  Time steps:         {len(ws)}")
    print(f"  WS range:           {np.nanmin(ws):.2f} -- {np.nanmax(ws):.2f} ft")

    assert not np.all(np.isnan(ws)), "All water surface values are NaN"
    assert max_flow > 0, "No flow at modified levee cross section"
else:
    print(f"")
    print(f"Cross section RS {modified_rs} not found in time series output.")
    print(f"Available stations (first 10): {stations[:10]}")

# Summary across all cross sections
peak_ws = xsec_ts["Maximum_Water_Surface"].values
peak_flow = xsec_ts["Maximum_Flow"].values
xs_summary = pd.DataFrame({
    "Station": stations,
    "Max WS (ft)": np.round(peak_ws, 2),
    "Max Flow (cfs)": np.round(peak_flow, 1),
})
print(f"")
print(f"Peak values across all {len(xs_summary)} cross sections:")
display(xs_summary.describe().round(2))

print(f"")
print(f"{'='*70}")
print(f"RESULTS EXTRACTION VALIDATED")
print(f"  Cross sections with data: {len(xs_summary)}")
print(f"  Global max water surface: {np.nanmax(peak_ws):.2f} ft")
fmt2 = f"{np.nanmax(peak_flow):,.1f}"
print(f"  Global max flow:          {fmt2} cfs")
print(f"{'='*70}")

## Summary

This notebook demonstrated:

- **`get_levees()`** -- Read levee data as a DataFrame with `xs_id`, `River`, `Reach`, `RS`, and station/elevation columns
- **`set_levees()`** -- Update, insert, or deactivate levee sides for individual cross sections
- **Filtering** -- Query by `xs_id` or `river`/`reach`/`rs` parameters
- **Round-trip verification** -- Read-modify-write with assertion checks
- **Batch analysis** -- Use DataFrame output for bulk statistics across geometry files
- **Cross-section visualization** -- Plot XS profiles with bank stations and levee positions; before/after comparison for the modified cross section
- **HEC-RAS compute validation** -- Modified geometry accepted by the unsteady processor
- **HDF results extraction** -- Cross-section water surface and flow time series extracted from simulation output

### Example Project

Bald Eagle Creek (1D Unsteady Flow Hydraulics) -- 189 cross sections, 41 with active levees.